In [2]:
import pandas as pd

df_ground_truth = pd.read_csv("ground_truth-new.csv")
ground_truth = df_ground_truth.to_dict(orient="records")

In [3]:
from pathlib import Path
import sys
import os

# 1. Get the absolute path of the directory containing this script
parent_dir = Path(os.getcwd()).resolve().parent

# 3. Add the parent directory to Python's search path
sys.path.append(str(parent_dir))

##############################################################################
# 4. Import the function normally
from ingest import load_faq_data, build_index

documents = load_faq_data()

documents_llm = []

for doc in documents:
    if doc["course"] == "llm-zoomcamp":
        documents_llm.append(doc)

documents = documents_llm
index = build_index(documents)

In [4]:
# Create a look-up table for the original FAQ documents:
doc_idx = {}

for doc in documents:
    doc_idx[doc["id"]] = doc

In [5]:
from dotenv import load_dotenv
from openai import OpenAI
import os

load_dotenv()
openai_client = OpenAI(
    api_key=os.getenv("GROQ_API_KEY"),
    base_url="https://api.groq.com/openai/v1"
)

In [ ]:
from evaluation_utils import RAGWithUsage

assistant = RAGWithUsage( # The assistant that uses the RAG approach with usage tracking
    index=index,
    llm_client=openai_client,
)

In [7]:
# Try running RAG for one question:
rec = ground_truth[0]
question = rec["question"]

answer_llm = assistant.rag(question)
answer_llm

'Yes, you can still join the course. If you want to receive a certificate, you’ll need to complete and submit your project while the submission window is still open.'

In [8]:
# Check the cost of this call:
assistant.total_cost()

0.0011655

In [9]:
# Get the original answer from document ID
doc_id = rec["document"]
original_doc = doc_idx[doc_id]
answer_orig = original_doc["answer"]

answer_orig

'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.'

In [10]:
# Now save both in one record:
rag_result = {
    "question": question,
    "answer_llm": answer_llm,
    "answer_orig": answer_orig,
    "document": doc_id,
}

rag_result

{'question': "Can I join the course now that it's already running?",
 'answer_llm': 'Yes, you can still join the course. If you want to receive a certificate, you’ll need to complete and submit your project while the submission window is still open.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

# Processing all questions:

In [11]:
# Encapsulate all we just did, creating a function that processes ONE ground truthrecord:
def generate_rag_answer(rec):
    question = rec["question"] # (The Q generated from A)
    doc_id = rec["document"]
    original_doc = doc_idx[doc_id]

    # answer from using RAG on the generated question (the A' answering Q)
    answer_llm = assistant.rag(question)
    # original answer from the FAQ documents (the A)
    answer_orig = original_doc["answer"]

    result = {
        "question": question,
        "answer_llm": answer_llm,
        "answer_orig": answer_orig,
        "document": doc_id,
    }

    return result

In [12]:
answer_record = generate_rag_answer(ground_truth[0])
answer_record

{'question': "Can I join the course now that it's already running?",
 'answer_llm': 'Yes – you can still join the course.\u202fIf you’d like to earn a certificate, just make sure to submit your capstone project while we’re still accepting submissions.',
 'answer_orig': 'Yes, but if you want to receive a certificate, you need to submit your project while we’re still accepting submissions.',
 'document': '74eb249bbf'}

In [13]:
assistant.reset_usage()

from concurrent.futures import ThreadPoolExecutor
from evaluation_utils import map_progress

with ThreadPoolExecutor(max_workers=6) as pool:
    results = map_progress(pool, ground_truth, generate_rag_answer)

answers = []

for answer_record in results:
    answers.append(answer_record)

assistant.total_cost()

df_answers = pd.DataFrame(answers)
df_answers.to_csv("data/rag-answers-new.csv", index=False)